**Compute Saturation**

In [3]:
#LSA = 16, Signal=K2, Detector = WD21
sg_id = 2
det_id = 3

**Fetch data from detectors**

In [5]:
import json
from datetime import datetime
from typing import List, Dict, Optional

class DetectorCountParser:
    
    def __init__(self, json_path: str):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
    
    def get_count_sum(self, 
                      detector_id: int, 
                      start_time: str, 
                      end_time: str) -> dict:
        """
        Berechnet die Summe der Counts für einen Detektor in einem beliebigen Zeitraum.
        
        Parameters:
            detector_id: ID des Detektors
            start_time: Startzeit im Format "HH:MM" (z.B. "06:00")
            end_time:   Endzeit im Format "HH:MM"   (z.B. "09:00")
        
        Returns:
            Dict mit Summe, Anzahl der Messungen und den einzelnen Werten
        """
        start_hour, start_minute = map(int, start_time.split(':'))
        end_hour, end_minute = map(int, end_time.split(':'))
        
        total_sum = 0
        count_entries = 0
        values = []
        
        for timeframe in self.data.get('timeFrames', []):
            timestamp_str = timeframe.get('timestamp')
            
            try:
                # Parse ISO Timestamp
                dt_str = timestamp_str.replace('Z', '+00:00')
                dt = datetime.fromisoformat(dt_str)
                
                # Zeitvergleich (nur Stunde und Minute)
                current_time = dt.time()
                start_t = datetime.strptime(start_time, "%H:%M").time()
                end_t = datetime.strptime(end_time, "%H:%M").time()
                
                # Prüfen ob aktueller Timestamp im gewünschten Bereich liegt
                if start_t <= current_time < end_t:
                    for detector in timeframe.get('detectors', []):
                        if detector.get('id') == detector_id:
                            count = detector.get('reading', {}) \
                                           .get('count', {}) \
                                           .get('value', 0)
                            
                            total_sum += count
                            count_entries += 1
                            values.append({
                                'timestamp': timestamp_str,
                                'count': count
                            })
                            break
            except Exception:
                continue  # Bei fehlerhaften Timestamps überspringen
        
        return {
            'detector_id': detector_id,
            'start_time': start_time,
            'end_time': end_time,
            'sum': total_sum,
            'measurements': count_entries,
            'average_per_measurement': round(total_sum / count_entries, 2) if count_entries > 0 else 0,
            'values': values
        }

In [22]:
parser = DetectorCountParser("detector/2026-04-01_0000.json")
result = parser.get_count_sum(detector_id=det_id, start_time="06:00", end_time="07:00")
print(result['sum'])        # ← die gewünschte Summe

demand = result['sum']

243


**Get Processdata from sipl stream**

In [18]:
import json
from datetime import datetime, timedelta
from typing import Dict


class SiplGreenTimeParser:
    """
    Parser für SI PL-Daten – Berechnet die exakte Grünzeit (Freigabezeit) 
    eines Signals in Sekunden.
    """
    
    def __init__(self, json_path: str):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        
        self.start_dt = datetime.fromisoformat(self.data['start'].replace('Z', '+00:00'))
        
        # Korrekte Zuordnung laut deiner Angabe:
        self.green_state = 4          # sgState 4 = Grün / Freigabe

    def get_green_seconds(self, 
                          signal_id: int, 
                          start_time: str = "06:00", 
                          end_time: str = "09:00") -> Dict:
        """
        Berechnet die Grünzeit (Freigabezeit) eines Signals in Sekunden.
        """
        start_t = datetime.strptime(start_time, "%H:%M").time()
        end_t = datetime.strptime(end_time, "%H:%M").time()

        total_green_ms = 0.0
        prev_dt = None

        for value in self.data.get('values', []):
            offset_ms = value.get('offset', 0)
            current_dt = self.start_dt + timedelta(milliseconds=offset_ms)
            current_time = current_dt.time()

            # Zeitfilter
            if not (start_t <= current_time < end_t):
                continue

            # Prüfen, ob das Signal Grün hat (sgState == 4)
            is_green = False
            for sig in value.get('sigState', []):
                if sig.get('id') == signal_id:
                    if sig.get('sgState') == self.green_state:
                        is_green = True
                    break

            # Delta-Zeit zum vorherigen Sample berechnen und bei Grün addieren
            if prev_dt is not None:
                delta_ms = (current_dt - prev_dt).total_seconds() * 1000
                if is_green:
                    total_green_ms += delta_ms

            prev_dt = current_dt

        total_seconds = round(total_green_ms / 1000)

        return {
            'signal_id': signal_id,
            'start_time': start_time,
            'end_time': end_time,
            'green_seconds': total_seconds,
            'green_minutes': round(total_seconds / 60, 2),
            'green_hours': round(total_seconds / 3600, 3)
        }

In [23]:
parser = SiplGreenTimeParser("signal/2026-04-01_0600.json")
result = parser.get_green_seconds(signal_id=sg_id, start_time="06:00", end_time="07:00")
print(result)

green_duration_seconds = result['green_seconds']

{'signal_id': 2, 'start_time': '06:00', 'end_time': '07:00', 'green_seconds': 1109, 'green_minutes': 18.48, 'green_hours': 0.308}


**Compute Saturation**

In [27]:
capacity = green_duration_seconds / 2
print(f"Kapazität (Fahrzeuge pro Stunde): {capacity}")

saturation = demand / capacity if capacity > 0 else float('inf')
print(f"Sättigung: {saturation:.2f}")

Kapazität (Fahrzeuge pro Stunde): 554.5
Sättigung: 0.44
